In [1]:
# !pip install langchain langchain_openai langchain_community pypdf faiss-cpu

In [2]:
# from google.colab import drive
# import os

# # 먼저 구글 드라이브 마운트
# drive.mount('/content/drive')

In [3]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")

# 환경 변수에서 API 키 가져오기
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
base_dir = Path(r"D:\WorkSpace\Python\langchain-tutorial")
file_path = base_dir / "Ch04. Advanced Rag" / "Data" / "투자설명서.pdf"
# file_path = (
#     "D:\WorkSpace\Python\langchain-tutorial\Ch04. Advanced Rag\Data\투자설명서.pdf"
# )
loader = PyPDFLoader(file_path)

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap = 200)

docs = loader.load_and_split(doc_splitter)

d:\WorkSpace\Python\langchain-tutorial\Ch04. Advanced Rag\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_openai.embeddings import OpenAIEmbeddings

# OpenAI의 임베딩 모델 사용
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [6]:
# FAISS 라이브러리 임포트
from langchain_community.vectorstores import FAISS

# FAISS DB 생성 후 저장
faiss_store = FAISS.from_documents(docs, embedding)
faiss_store.save_local("/content/DB")

In [7]:
# 저장된 DB경로 지정 후, DB 로드
persist_directory = "/content/DB"
vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

In [8]:
# FAISS 리트리버 생성
faiss_retriever = vectordb.as_retriever(search_kwargs={"k": 2})

In [9]:
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_openai import ChatOpenAI

# 관련있는 문서를 수집 후, Chatgpt로 최종 답변까지 수행
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0.2, model="gpt-4o"),
    chain_type="stuff",
    retriever=faiss_retriever,
    return_source_documents=True # 답변에 사용된 source document도 보여주도록 설정
)

In [10]:
qa_chain.invoke("이 회사가 발행한 주식의 총 발행량이 어느정도야?")

{'query': '이 회사가 발행한 주식의 총 발행량이 어느정도야?',
 'result': '죄송하지만, 제공된 정보에서는 회사가 발행한 주식의 총 발행량에 대한 구체적인 숫자가 명시되어 있지 않습니다.',
 'source_documents': [Document(id='d26f9d54-1dc1-4c9c-a619-88f64eb079f8', metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2024-06-26T16:15:14+09:00', 'moddate': '2024-06-26T16:15:14+09:00', 'source': 'D:\\WorkSpace\\Python\\langchain-tutorial\\Ch04. Advanced Rag\\Data\\투자설명서.pdf', 'total_pages': 514, 'page': 327, 'page_label': '328'}, page_content="(*1) 정부R&D예산 축소로 인한 사업비 규모가 변경되었습니다. 기타 세부 사항은 2024년\n3월 15일 기 제출한 기타경영사항을 참조하시길 바랍니다.  \n \n(*2) 당사는 2024년 3월 12일 이사회 결의에 의거하여 주식회사 포베이커를 소규모 합병의\n형태로 흡수합병을 결정하였습니다. \xa02024년 5월 14일을 합병기일로 하여 흡수합병을 완료\n하였습니다. 자세한 내용은 2024년 3월 12일 '주요사항보고서(회사합병결정)' 과 \xa02024년\n5월 21일 '합병등종료보고서(합병)' 공시를 참고하시기 바랍니다. \n  \n3. 자본금 변동사항 \n \n당사는 본 보고서 작성기준일 현재 기업공시서식 작성기준에 따른 소규모 기업에 해당하므\n로 상기 자본금 변동추이를 기재하지 않았습니다. \n  \n4. 주식의 총수 등 \n  \n가. 주식의 총수 \n  \n주식의 총수 현황 \n설\n투자 차입금에 대